# DEMO PIPELINE — 02 SILVER
## Bronze → cleaned and typed Silver

Silver goal:

```text
clean
standardize
cast data types
validate
```

Input:

```text
demo_orders_bronze
126 rows
```

Output:

```text
demo_orders_silver
126 rows
```


In [ ]:
# ============================================================
# SETUP — RESOLVE CATALOG / SCHEMAS / OBJECT NAMES
# ============================================================
# WHAT:
#   Rebuild the same names used by the previous notebook.
#
# WHY:
#   Variables do not automatically persist between separate notebooks.
#
# EXPECTED:
#   All layer object names resolve consistently.

ctx = spark.sql("""
SELECT
    current_catalog() AS catalog,
    current_schema() AS schema
""").first()

CATALOG = ctx["catalog"]
CURRENT_SCHEMA = ctx["schema"]

if CATALOG == "hive_metastore":
    raise RuntimeError(
        "Switch to a Unity Catalog catalog first."
    )

def qi(value: str) -> str:
    return f"`{value.replace('`', '``')}`"

available = {
    r["databaseName"]
    for r in spark.sql(
        f"SHOW SCHEMAS IN {qi(CATALOG)}"
    ).collect()
}

BRONZE_SCHEMA = (
    "retail_bronze"
    if "retail_bronze" in available
    else CURRENT_SCHEMA
)

SILVER_SCHEMA = (
    "retail_silver"
    if "retail_silver" in available
    else CURRENT_SCHEMA
)

GOLD_SCHEMA = (
    "retail_gold"
    if "retail_gold" in available
    else CURRENT_SCHEMA
)

SOURCE_FILE = (
    f"/Volumes/{CATALOG}/{BRONZE_SCHEMA}/"
    f"demo_source_files/orders/orders.csv"
)

BRONZE_TABLE = (
    f"{qi(CATALOG)}.{qi(BRONZE_SCHEMA)}."
    f"{qi('demo_orders_bronze')}"
)

SILVER_TABLE = (
    f"{qi(CATALOG)}.{qi(SILVER_SCHEMA)}."
    f"{qi('demo_orders_silver')}"
)

GOLD_TABLE = (
    f"{qi(CATALOG)}.{qi(GOLD_SCHEMA)}."
    f"{qi('demo_order_status_gold')}"
)

print("Source :", SOURCE_FILE)
print("Bronze :", BRONZE_TABLE)
print("Silver :", SILVER_TABLE)
print("Gold   :", GOLD_TABLE)


In [ ]:
# ============================================================
# CELL 2 — READ BRONZE
# ============================================================
# WHAT:
#   Read the managed Bronze Delta table.
#
# WHY:
#   Silver should always build from Bronze, not go back to the raw source file.
#
# EXPECTED:
#   126 Bronze rows are available.

from pyspark.sql import functions as F

bronze_df = spark.table(
    BRONZE_TABLE
)

assert bronze_df.count() == 126

display(
    bronze_df.limit(10)
)


In [ ]:
# ============================================================
# CELL 3 — CLEAN AND STANDARDIZE THE ORDERS
# ============================================================
# WHAT:
#   Convert timestamp, normalize status/city/channel and trim IDs.
#
# WHY:
#   Silver should provide consistent business-ready columns.
#
# INPUT GRAIN:
#   One row = one Bronze order.
#
# OUTPUT GRAIN:
#   One row = one cleaned order.
#
# EXPECTED:
#   Row count remains 126.

silver_df = (
    bronze_df
    .select(
        F.trim(
            "order_id"
        ).alias(
            "order_id"
        ),
        F.trim(
            "customer_id"
        ).alias(
            "customer_id"
        ),
        F.to_timestamp(
            "order_ts"
        ).alias(
            "order_ts"
        ),
        F.upper(
            F.trim(
                "order_status"
            )
        ).alias(
            "order_status"
        ),
        F.initcap(
            F.trim(
                "shipping_city"
            )
        ).alias(
            "shipping_city"
        ),
        F.upper(
            F.trim(
                "channel"
            )
        ).alias(
            "channel"
        ),
        "source_file",
        "ingestion_ts",
    )
    .filter(
        F.col("order_id").isNotNull()
        & F.col("customer_id").isNotNull()
        & F.col("order_ts").isNotNull()
    )
)

assert silver_df.count() == 126

display(
    silver_df.limit(10)
)


In [ ]:
# ============================================================
# CELL 4 — WRITE THE SILVER DELTA TABLE
# ============================================================
# WHAT:
#   Persist cleaned orders as a managed Delta table.
#
# WHY:
#   Gold should consume a stable, standardized Silver layer.
#
# EXPECTED:
#   demo_orders_silver contains 126 rows.

(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .saveAsTable(
        SILVER_TABLE
    )
)

silver_count = spark.table(
    SILVER_TABLE
).count()

print("Silver rows:", silver_count)

assert silver_count == 126


In [ ]:
# ============================================================
# CELL 5 — VALIDATE SILVER STATUS COUNTS
# ============================================================
# WHAT:
#   Check official order-status totals.
#
# WHY:
#   Cleaning must not accidentally change business counts.
#
# EXPECTED:
#   DELIVERED=78, SHIPPED=29, PLACED=10, CANCELLED=9.

status_df = (
    spark.table(
        SILVER_TABLE
    )
    .groupBy(
        "order_status"
    )
    .count()
    .orderBy(
        F.col("count").desc()
    )
)

display(status_df)

status_counts = {
    row["order_status"]: row["count"]
    for row in status_df.collect()
}

assert status_counts == {
    "DELIVERED": 78,
    "SHIPPED": 29,
    "PLACED": 10,
    "CANCELLED": 9,
}

print("\nSILVER: PASSED")
print("Next notebook: 03_GOLD")
